In [10]:
import os
import pandas as pd
from sqlalchemy import create_engine
import pyodbc
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
# import pyarrow as pa
# import pyarrow.csv as csv
from datetime import date, timedelta

In [11]:
# Получаем сегодняшнюю дату
today = date.today() - timedelta(days=3)

# Создаем список из 7 дат, начиная с сегодняшней
last_week = [(today - timedelta(days=i)) for i in range(7)]

In [12]:
last_week[0].strftime(format='%Y-%m-%d')
last_week[6].strftime(format='%Y-%m-%d')

'2026-04-06'

In [13]:
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    Engine = create_engine(connection_string)
    return Engine
SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [14]:
Engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)
query_distribution = f"""
WITH ref AS (
    SELECT DISTINCT
        q.itemid                                AS [Артикул],
        b.inventsizeid                          AS [Размер],
        q.businessgroupru                       AS [Бизнес-группа],
        q.DEPARTMENTIDRU                        AS [Розничный отдел],
        CONCAT(q.retailgroup,' ',q.grpnameru)   AS [Группа],
        q.KAR_SEASONCODERU                      AS [Сезон],
        q.trademark                             AS [Бренд],
        q.buyer                                 AS [Ответственный за группу],
        q.KAR_ACTUALCOLLECTION                  AS [Коллекция]
    FROM [DBReport].[dbo].[GuideAssortiment] q
    INNER JOIN [DynamicsAx1].[dbo].[INVENTITEMBARCODE] a
        ON q.itemid = a.itemid AND a.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTDIM] b
        ON a.inventdimid = b.inventdimid AND b.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTTABLE] c
        ON q.itemid = c.itemid AND c.dataareaid = 'vrt' AND c.itemgroupid = 'Goods'
    WHERE q.businessgroupru IN (N'Одежда для детей', N'Одежда и аксессуары')
      AND q.KAR_ACTUALCOLLECTION = '2026SS'
),

total_sizes AS (
    SELECT
        [Артикул],
        COUNT(DISTINCT CASE
            WHEN LTRIM(RTRIM([Размер])) <> '' THEN [Размер]
        END) AS [Всего размеров]
    FROM ref
    GROUP BY [Артикул]
),

attrs AS (
    SELECT
        [Артикул],
        MAX([Бизнес-группа])           AS [Бизнес-группа],
        MAX([Розничный отдел])         AS [Розничный отдел],
        MAX([Группа])                  AS [Группа],
        MAX([Сезон])                   AS [Сезон],
        MAX([Бренд])                   AS [Бренд],
        MAX([Ответственный за группу]) AS [Ответственный за группу],
        MAX([Коллекция])               AS [Коллекция]
    FROM ref
    GROUP BY [Артикул]
),

wb AS (
    SELECT ITEMID, MIN(NMID) AS [Артикул WB]
    FROM [DBPartners].[dbo].[WblmRepGetNomenclatureWildberries]
    GROUP BY ITEMID
),

articles_in_period AS (
    SELECT DISTINCT itemid AS [Артикул]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries]
    WHERE dt BETWEEN '{last_week[6].strftime(format='%Y-%m-%d')}' AND '{last_week[0].strftime(format='%Y-%m-%d')}'
      AND qte > 0
),

stock_12 AS (
    SELECT
        s.itemid        AS [Артикул],
        SUM(s.qte)      AS [Остаток на 12.04]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{last_week[0].strftime(format='%Y-%m-%d')}'
    GROUP BY s.itemid
),

agg_sizes AS (
    SELECT
        s.itemid                        AS [Артикул],
        COUNT(DISTINCT s.INVENTSIZEID)  AS [Размеров на агрегаторе]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{last_week[0].strftime(format='%Y-%m-%d')}'
      AND s.qte > 0
    GROUP BY s.itemid
),

marketing AS (
    SELECT
        a.[Артикул WB]          AS [Артикул WB],
        SUM(a.[Показы])         AS [Показы],
        SUM(a.[Кол-во переходов в карточку товара])  AS [Клики],
        SUM(a.[Заказали товаров, шт]) AS [Заказы]
    FROM [DBReport].[mp].[wb_sales_funnel_lk] a
    WHERE a.[Дата] BETWEEN '{last_week[6].strftime(format='%Y-%m-%d')}' AND '{last_week[0].strftime(format='%Y-%m-%d')}'
    GROUP BY a.[Артикул WB]
)

SELECT
    a.[Артикул],
    wb.[Артикул WB],
    a.[Бизнес-группа],
    a.[Розничный отдел],
    a.[Группа],
    a.[Сезон],
    a.[Бренд],
    a.[Ответственный за группу],
    a.[Коллекция],
    ISNULL(m.[Показы], 0)               AS [Показы],
    ISNULL(m.[Клики], 0)                 AS [Клики],
    ISNULL(m.[Заказы], 0)                AS [Заказы],
    CAST(CASE
        WHEN ISNULL(m.[Показы], 0) = 0 THEN 0
        ELSE m.[Клики] * 100.0 / m.[Показы]
    END AS DECIMAL(5, 2))                AS [CTR, %],
    CAST(CASE
        WHEN ISNULL(m.[Клики], 0) = 0 THEN 0
        ELSE m.[Заказы] * 100.0 / m.[Клики]
    END AS DECIMAL(5, 2))                AS [Конверсия клики в заказы, %],
    ISNULL(st.[Остаток на 12.04], 0)     AS [Остаток на '{last_week[0].strftime(format='%d.%m')}'],
    CAST(CASE
        WHEN ISNULL(ts.[Всего размеров], 0) = 0 THEN 0
        ELSE ISNULL(ag.[Размеров на агрегаторе], 0) * 100.0 / ts.[Всего размеров]
    END AS DECIMAL(5, 2))                AS [Дистрибуция, %]
FROM articles_in_period ap
INNER JOIN attrs a       ON ap.[Артикул] = a.[Артикул]
LEFT JOIN wb             ON a.[Артикул] = wb.ITEMID
LEFT JOIN total_sizes ts ON a.[Артикул] = ts.[Артикул]
LEFT JOIN agg_sizes ag   ON a.[Артикул] = ag.[Артикул]
LEFT JOIN stock_12 st    ON a.[Артикул] = st.[Артикул]
LEFT JOIN marketing m    ON wb.[Артикул WB] = m.[Артикул WB]
ORDER BY a.[Артикул]

"""
df_base = pd.read_sql(query_distribution, Engine)

'{last_week[6].strftime(format='%Y-%m-%d')}' AND '{last_week[0].strftime(format='%Y-%m-%d')}'

In [16]:
df_base['Показы'].sum()

np.int64(24381449)

In [ ]:
def build_query(start_date, end_date) -> str:
    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')
    end_label = end_date.strftime('%d.%m')
    # bg_list = ", ".join(f"N'{g}'" for g in config.BUSINESS_GROUPS)

    return f"""
WITH ref AS (
    SELECT DISTINCT
        q.itemid                                AS [Артикул],
        b.inventsizeid                          AS [Размер],
        q.businessgroupru                       AS [Бизнес-группа],
        q.DEPARTMENTIDRU                        AS [Розничный отдел],
        CONCAT(q.retailgroup,' ',q.grpnameru)   AS [Группа],
        q.KAR_SEASONCODERU                      AS [Сезон],
        q.trademark                             AS [Бренд],
        q.buyer                                 AS [Ответственный за группу],
        q.KAR_ACTUALCOLLECTION                  AS [Коллекция]
    FROM [DBReport].[dbo].[GuideAssortiment] q
    INNER JOIN [DynamicsAx1].[dbo].[INVENTITEMBARCODE] a
        ON q.itemid = a.itemid AND a.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTDIM] b
        ON a.inventdimid = b.inventdimid AND b.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTTABLE] c
        ON q.itemid = c.itemid AND c.dataareaid = 'vrt' AND c.itemgroupid = 'Goods'
    WHERE q.businessgroupru IN (N'Одежда для детей', N'Одежда и аксессуары')
      AND q.KAR_ACTUALCOLLECTION = '{config.COLLECTION}'
),
total_sizes AS (
    SELECT [Артикул],
        COUNT(DISTINCT CASE WHEN LTRIM(RTRIM([Размер])) <> '' THEN [Размер] END) AS [Всего размеров]
    FROM ref GROUP BY [Артикул]
),
attrs AS (
    SELECT [Артикул],
        MAX([Бизнес-группа]) AS [Бизнес-группа],
        MAX([Розничный отдел]) AS [Розничный отдел],
        MAX([Группа]) AS [Группа],
        MAX([Сезон]) AS [Сезон],
        MAX([Бренд]) AS [Бренд],
        MAX([Ответственный за группу]) AS [Ответственный за группу],
        MAX([Коллекция]) AS [Коллекция]
    FROM ref GROUP BY [Артикул]
),
wb AS (
    SELECT ITEMID, MIN(NMID) AS [Артикул WB]
    FROM [DBPartners].[dbo].[WblmRepGetNomenclatureWildberries]
    GROUP BY ITEMID
),
articles_in_period AS (
    SELECT DISTINCT itemid AS [Артикул]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries]
    WHERE dt BETWEEN '{start_str}' AND '{end_str}'
      AND qte > 0
),
stock_end AS (
    SELECT s.itemid AS [Артикул], SUM(s.qte) AS [Остаток]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{end_str}'
    GROUP BY s.itemid
),
agg_sizes AS (
    SELECT s.itemid AS [Артикул], COUNT(DISTINCT s.INVENTSIZEID) AS [Размеров на агрегаторе]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{end_str}' AND s.qte > 0
    GROUP BY s.itemid
),
marketing AS (
    SELECT
        a.[Артикул WB]          AS [Артикул WB],
        SUM(a.[Показы])         AS [Показы],
        SUM(a.[Кол-во переходов в карточку товара])  AS [Клики],
        SUM(a.[Заказали товаров, шт]) AS [Заказы]
    FROM [DBReport].[mp].[wb_sales_funnel_lk] a
    WHERE a.[Дата] BETWEEN '{start_str}' AND '{end_str}'
    GROUP BY a.[Артикул WB]
)
SELECT
    a.[Артикул],
    wb.[Артикул WB],
    a.[Бизнес-группа],
    a.[Розничный отдел],
    a.[Группа],
    a.[Сезон],
    a.[Бренд],
    a.[Ответственный за группу],
    a.[Коллекция],
    ISNULL(m.[Показы], 0) AS [Показы],
    ISNULL(m.[Клики], 0) AS [Клики],
    ISNULL(m.[Заказы], 0) AS [Заказы],
    CAST(CASE WHEN ISNULL(m.[Показы], 0) = 0 THEN 0
         ELSE m.[Клики] * 100.0 / m.[Показы] END AS DECIMAL(5, 2)) AS [CTR, %],
    CAST(CASE WHEN ISNULL(m.[Клики], 0) = 0 THEN 0
         ELSE m.[Заказы] * 100.0 / m.[Клики] END AS DECIMAL(5, 2)) AS [Конверсия клики в заказы, %],
    ISNULL(st.[Остаток], 0) AS [Остаток на {end_label}],
    CAST(CASE WHEN ISNULL(ts.[Всего размеров], 0) = 0 THEN 0
         ELSE ISNULL(ag.[Размеров на агрегаторе], 0) * 100.0 / ts.[Всего размеров]
         END AS DECIMAL(5, 2)) AS [Дистрибуция, %]
FROM articles_in_period ap
INNER JOIN attrs a       ON ap.[Артикул] = a.[Артикул]
LEFT JOIN wb             ON a.[Артикул] = wb.ITEMID
LEFT JOIN total_sizes ts ON a.[Артикул] = ts.[Артикул]
LEFT JOIN agg_sizes ag   ON a.[Артикул] = ag.[Артикул]
LEFT JOIN stock_end st   ON a.[Артикул] = st.[Артикул]
LEFT JOIN marketing m    ON wb.[Артикул WB] = m.[Артикул WB]
ORDER BY a.[Артикул]
"""


def load_base_dataframe(start_date, end_date) -> pd.DataFrame:
    """Выполняет SQL-запрос и возвращает df_base."""
    engine = connect_to_sql(config.SQL_SERVER, config.SQL_DB_PARTNERS)
    query = build_query(start_date, end_date)
    df = pd.read_sql(query, engine)
    df.to_excel('Query result.xlsx')
    return df

In [ ]:
def build_query(start_date, end_date) -> str:
    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')
    end_label = end_date.strftime('%d.%m')
    # bg_list = ", ".join(f"N'{g}'" for g in config.BUSINESS_GROUPS)

    return f"""
WITH ref AS (
    SELECT DISTINCT
        q.itemid                                AS [Артикул],
        b.inventsizeid                          AS [Размер],
        q.businessgroupru                       AS [Бизнес-группа],
        q.DEPARTMENTIDRU                        AS [Розничный отдел],
        CONCAT(q.retailgroup,' ',q.grpnameru)   AS [Группа],
        q.KAR_SEASONCODERU                      AS [Сезон],
        q.trademark                             AS [Бренд],
        q.buyer                                 AS [Ответственный за группу],
        q.KAR_ACTUALCOLLECTION                  AS [Коллекция]
    FROM [DBReport].[dbo].[GuideAssortiment] q
    INNER JOIN [DynamicsAx1].[dbo].[INVENTITEMBARCODE] a
        ON q.itemid = a.itemid AND a.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTDIM] b
        ON a.inventdimid = b.inventdimid AND b.dataareaid = 'vrt'
    INNER JOIN [DynamicsAx1].[dbo].[INVENTTABLE] c
        ON q.itemid = c.itemid AND c.dataareaid = 'vrt' AND c.itemgroupid = 'Goods'
    WHERE q.businessgroupru IN (N'Одежда для детей', N'Одежда и аксессуары')
      AND q.KAR_ACTUALCOLLECTION = '{config.COLLECTION}'
),
total_sizes AS (
    -- Одноразмерные товары (сумки, ремни, аксессуары) хранятся с NULL/пустым
    -- inventsizeid. Если после фильтрации "непустых" размеров COUNT=0, считаем
    -- как 1 (товар существует в одном варианте). Аналог Python-логики
    -- `len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1`.
    SELECT [Артикул],
        ISNULL(
            NULLIF(
                COUNT(DISTINCT CASE WHEN LTRIM(RTRIM([Размер])) <> '' THEN [Размер] END),
                0
            ),
            1
        ) AS [Всего размеров]
    FROM ref GROUP BY [Артикул]
),
attrs AS (
    SELECT [Артикул],
        MAX([Бизнес-группа]) AS [Бизнес-группа],
        MAX([Розничный отдел]) AS [Розничный отдел],
        MAX([Группа]) AS [Группа],
        MAX([Сезон]) AS [Сезон],
        MAX([Бренд]) AS [Бренд],
        MAX([Ответственный за группу]) AS [Ответственный за группу],
        MAX([Коллекция]) AS [Коллекция]
    FROM ref GROUP BY [Артикул]
),
wb AS (
    SELECT ITEMID, MIN(NMID) AS [Артикул WB]
    FROM [DBPartners].[dbo].[WblmRepGetNomenclatureWildberries]
    GROUP BY ITEMID
),
articles_in_period AS (
    -- Артикул попадает в отчёт, если он:
    --   (1) имел qte > 0 на WB в период (реально был в продаже), ИЛИ
    --   (2) имел маркетинговые события (показы/клики/заказы) за период —
    --       даже если на складе ничего нет (распродан, но карточка активна).
    -- Раньше использовался только (1), из-за чего терялись распроданные товары
    -- с активным маркетингом (часто у них показов больше всего).
    SELECT DISTINCT itemid AS [Артикул]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries]
    WHERE dt BETWEEN '{start_str}' AND '{end_str}'
      AND qte > 0
    UNION
    SELECT DISTINCT wb_n.ITEMID AS [Артикул]
    FROM [DBReport].[mp].[wb_sales_funnel_lk] mk
    INNER JOIN [DBPartners].[dbo].[WblmRepGetNomenclatureWildberries] wb_n
        ON wb_n.NMID = mk.[Артикул WB]
    WHERE mk.[Дата] BETWEEN '{start_str}' AND '{end_str}'
      AND wb_n.ITEMID IS NOT NULL
),
stock_end AS (
    SELECT s.itemid AS [Артикул],
           ISNULL(SUM(ISNULL(s.qte, 0)), 0) AS [Остаток]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{end_str}'
    GROUP BY s.itemid
),
agg_sizes AS (
    -- Для одноразмерных товаров INVENTSIZEID может быть NULL/'' → COUNT DISTINCT=0.
    -- Но если товар присутствует в stock с qte>0, значит "1 размер" на агрегаторе.
    -- Поэтому 0 → 1 (такая же защита как в total_sizes).
    SELECT s.itemid AS [Артикул],
        ISNULL(
            NULLIF(
                COUNT(DISTINCT CASE WHEN LTRIM(RTRIM(ISNULL(s.INVENTSIZEID, ''))) <> ''
                                    THEN s.INVENTSIZEID END),
                0
            ),
            1
        ) AS [Размеров на агрегаторе]
    FROM [DBPartners].[dbo].[WblmRepGetStockWildberries] s
    WHERE s.dt = '{end_str}' AND s.qte > 0
    GROUP BY s.itemid
),
marketing AS (
    -- ISNULL вокруг агрегатов, чтобы не ловить warning
    -- "Null value is eliminated by an aggregate or other SET operation"
    SELECT
        a.[Артикул WB]                                      AS [Артикул WB],
        ISNULL(SUM(ISNULL(a.[Показы], 0)), 0)               AS [Показы],
        ISNULL(SUM(ISNULL(a.[Кол-во переходов в карточку товара], 0)), 0) AS [Клики],
        ISNULL(SUM(ISNULL(a.[Заказали товаров, шт], 0)), 0) AS [Заказы]
    FROM [DBReport].[mp].[wb_sales_funnel_lk] a
    WHERE a.[Дата] BETWEEN '{start_str}' AND '{end_str}'
    GROUP BY a.[Артикул WB]
)
SELECT
    a.[Артикул],
    wb.[Артикул WB],
    a.[Бизнес-группа],
    a.[Розничный отдел],
    a.[Группа],
    a.[Сезон],
    a.[Бренд],
    a.[Ответственный за группу],
    a.[Коллекция],
    ISNULL(m.[Показы], 0) AS [Показы],
    ISNULL(m.[Клики], 0) AS [Клики],
    ISNULL(m.[Заказы], 0) AS [Заказы],
    -- DECIMAL(9,2) вместо (5,2): (5,2) вмещает максимум 999.99, а Конверсия
    -- при данных wb_sales_funnel_lk (Заказы могут быть много больше Кликов
    -- из-за многоштучных корзин) иногда выходит за эту границу → overflow.
    -- (9,2) поддерживает значения до 9,999,999.99 — с большим запасом.
    CAST(CASE WHEN ISNULL(m.[Показы], 0) = 0 THEN 0
         ELSE ISNULL(m.[Клики], 0) * 100.0 / m.[Показы] END AS DECIMAL(9, 2)) AS [CTR, %],
    CAST(CASE WHEN ISNULL(m.[Клики], 0) = 0 THEN 0
         ELSE ISNULL(m.[Заказы], 0) * 100.0 / m.[Клики] END AS DECIMAL(9, 2)) AS [Конверсия клики в заказы, %],
    ISNULL(st.[Остаток], 0) AS [Остаток на {end_label}],
    CAST(CASE WHEN ISNULL(ts.[Всего размеров], 0) = 0 THEN 0
         ELSE ISNULL(ag.[Размеров на агрегаторе], 0) * 100.0 / ts.[Всего размеров]
         END AS DECIMAL(9, 2)) AS [Дистрибуция, %]
FROM articles_in_period ap
INNER JOIN attrs a       ON ap.[Артикул] = a.[Артикул]
LEFT JOIN wb             ON a.[Артикул] = wb.ITEMID
LEFT JOIN total_sizes ts ON a.[Артикул] = ts.[Артикул]
LEFT JOIN agg_sizes ag   ON a.[Артикул] = ag.[Артикул]
LEFT JOIN stock_end st   ON a.[Артикул] = st.[Артикул]
LEFT JOIN marketing m    ON wb.[Артикул WB] = m.[Артикул WB]
ORDER BY a.[Артикул]
"""

In [ ]:
# === ЗАГРУЗКА КОЛИЧЕСТВА ФОТО ЧЕРЕЗ WB API (BATCH) + КЭШ ===
#
# WB Content API v2 НЕ поддерживает массив nmID в одном запросе.
# Cursor-пагинация по каталогу — самый быстрый легитимный метод.
#
# Поведение кэша (photo_cache.xlsx):
#   USE_PHOTO_CACHE=True  + кэш есть  → читаем из кэша, без API
#   USE_PHOTO_CACHE=True  + кэша нет  → API + СОХРАНЕНИЕ кэша
#   USE_PHOTO_CACHE=False             → API + СОХРАНЕНИЕ кэша
# Кэш ВСЕГДА перезаписывается после обращения к API.

import sys
sys.path.insert(0, '.')
from wb_otbor.wb_api import get_photo_counts

USE_PHOTO_CACHE = False   # ← поставьте True для отладки (без запроса к WB)

nm_ids_list = df_base['Артикул WB'].dropna().unique().tolist()
photo_counts = get_photo_counts(
    nm_ids_list,
    use_cache=USE_PHOTO_CACHE,
    log=print,
)

# === ДОБАВЛЯЕМ СТОЛБЕЦ (raw count - 2, минимум 0) ===
def get_photo_count_minus2(wb_id):
    if pd.isna(wb_id) or str(wb_id).strip() == '':
        return 0
    key = str(int(float(wb_id)))
    raw_count = photo_counts.get(key, 0)
    return max(raw_count - 2, 0)

df_base['Количество фото (-2 от скрипта)'] = df_base['Артикул WB'].apply(get_photo_count_minus2)

# === СОРТИРОВКА ПО ПОКАЗАМ ПО УБЫВАНИЮ ===
df_base = df_base.sort_values('Показы', ascending=False).reset_index(drop=True)

print(f"
Пример (после сортировки по Показам DESC):")
print(df_base[['Артикул', 'Артикул WB', 'Показы', 'Количество фото (-2 от скрипта)']].head(10))


In [8]:
df_base

,Артикул,Артикул WB,Бизнес-группа,Розничный отдел,Группа,Сезон,Бренд,Ответственный за группу,Коллекция,Показы,Клики,Заказы,"CTR, %","Конверсия клики в заказы, %",Остаток на '12.04',"Дистрибуция, %",Количество фото (-2 от скрипта)
0,07200720,895955626,Одежда и аксессуары,Нижнее белье женское,072 Трусы женские,всесезонный,kari,Коновалова А.,2026SS,6895,256,0,3.71,0.00,1648,100.0,18
1,07200730,936940604,Одежда и аксессуары,Нижнее белье женское,072 Трусы женские,всесезонный,kari,Коновалова А.,2026SS,225,24,0,10.67,0.00,600,80.0,10
2,07200740,901030050,Одежда и аксессуары,Нижнее белье женское,072 Трусы женские,всесезонный,kari,Коновалова А.,2026SS,14050,405,0,2.88,0.00,1850,100.0,16
3,07200750,895955627,Одежда и аксессуары,Нижнее белье женское,072 Трусы женские,всесезонный,kari,Коновалова А.,2026SS,6661,536,2,8.05,0.37,1848,100.0,21
4,07200760,936940644,Одежда и аксессуары,Нижнее белье женское,072 Трусы женские,всесезонный,kari,Коновалова А.,2026SS,252,6,0,2.38,0.00,150,40.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1318,y9708260,321683802,Одежда и аксессуары,"Футболки, рубашки мужские",h10 Футболка мужская,всесезонный,kari,,2026SS,13598,2771,91,20.38,3.28,511,100.0,11
1319,y9708270,321683759,Одежда и аксессуары,"Футболки, рубашки мужские",h10 Футболка мужская,всесезонный,kari,,2026SS,47534,4564,199,9.60,4.36,68,80.0,10
1320,y9708290,321683812,Одежда и аксессуары,"Футболки, рубашки мужские",h10 Футболка мужская,всесезонный,kari,,2026SS,14942,3283,189,21.97,5.76,48,80.0,11
1321,y9808010,378959433,Одежда и аксессуары,Спортивная одежда для мужчин,y98 Шорты мужские,всесезонный,kari,Коновалова И. Одежда,2026SS,1518,79,0,5.20,0.00,573,100.0,4


In [9]:
from openpyxl import load_workbook
from copy import copy

# === НАСТРОЙКИ ===
template_path = 'Отбор 13.04.xlsx'
output_date = last_week[0].strftime('%d.%m')
output_path = f'Отбор {output_date}.xlsx'

# === ЗАГРУЗКА ШАБЛОНА ===
wb = load_workbook(template_path)
ws = wb.active

# === СОХРАНЯЕМ СТИЛИ ИЗ СТРОКИ 3 (шаблонная строка данных) ===
row3_styles = {}
for col in range(1, 25):
    cell = ws.cell(row=3, column=col)
    row3_styles[col] = {
        'font': copy(cell.font),
        'fill': copy(cell.fill),
        'border': copy(cell.border),
        'alignment': copy(cell.alignment),
        'number_format': cell.number_format,
    }

# === ОЧИЩАЕМ ВСЕ СТРОКИ ДАННЫХ (3 до конца) ===
for row in range(3, ws.max_row + 1):
    for col in range(1, 25):
        ws.cell(row=row, column=col).value = None

# === ОПРЕДЕЛЯЕМ ИМЕНА СТОЛБЦОВ ИЗ df_base ===
остаток_col = [c for c in df_base.columns if 'Остаток' in c][0]
distrib_col = [c for c in df_base.columns if 'Дистрибуция' in c][0]

last_data_row = 2 + len(df_base)

# === ЗАПОЛНЯЕМ ДАННЫЕ ===
for idx, row_data in df_base.iterrows():
    r = 3 + idx

    # A-I: справочные данные (значения)
    ws.cell(row=r, column=1).value  = str(row_data['Артикул'])
    ws.cell(row=r, column=2).value  = str(int(row_data['Артикул WB'])) if pd.notna(row_data['Артикул WB']) else ''
    ws.cell(row=r, column=3).value  = row_data['Бизнес-группа']
    ws.cell(row=r, column=4).value  = row_data['Розничный отдел']
    ws.cell(row=r, column=5).value  = row_data['Группа']
    ws.cell(row=r, column=6).value  = row_data['Сезон']
    ws.cell(row=r, column=7).value  = row_data['Бренд']
    ws.cell(row=r, column=8).value  = row_data['Ответственный за группу']
    ws.cell(row=r, column=9).value  = row_data['Коллекция']

    # J-L: маркетинг (значения)
    ws.cell(row=r, column=10).value = int(row_data['Показы'])
    ws.cell(row=r, column=11).value = int(row_data['Клики'])
    ws.cell(row=r, column=12).value = int(row_data['Заказы'])

    # M: CTR (формула, формат 0.00%)
    ws.cell(row=r, column=13).value = f'=IFERROR(K{r}/J{r},0)'

    # N: Конверсия (формула, формат 0.00%)
    ws.cell(row=r, column=14).value = f'=IFERROR(L{r}/K{r},0)'

    # O: Остаток (значение)
    ws.cell(row=r, column=15).value = int(row_data[остаток_col])

    # P: Дистрибуция (значение, делим на 100 т.к. формат ячейки 0.0%)
    ws.cell(row=r, column=16).value = row_data[distrib_col] / 100

    # --- Определяем условия для Q (Техничка) и R (Отбор для поиска) ---
    stock   = row_data[остаток_col]
    distrib = row_data[distrib_col]
    shows   = row_data['Показы']

    # R (col 18): Отбор для поиска — перекрываем формулу только при попадании в условие
    if stock < 10 or distrib < 20:
        ws.cell(row=r, column=18).value = 'Мало остатка'
        ws.cell(row=r, column=17).value = 0  # Q: Техничка
    elif shows < 1200:
        ws.cell(row=r, column=18).value = 'Мало показов'
        ws.cell(row=r, column=17).value = 0  # Q: Техничка
    else:
        # Формула остаётся нетронутой
        ws.cell(row=r, column=18).value = f'=IF(OR(M{r}<$S$1*0.5,N{r}<$T$1*0.5),"код для проверки","нормальный код")'
        ws.cell(row=r, column=17).value = 1  # Q: Техничка

    # S (col 19): Отбор по CTR
    ws.cell(row=r, column=19).value = f'=IF(M{r}<$S$1*0.5,"Низшая четверть",IF(M{r}<$S$1,"Вторая четверть","Больше половины"))'

    # T (col 20): Отбор по CR
    ws.cell(row=r, column=20).value = f'=IF(N{r}<$T$1*0.5,"Низшая четверть",IF(N{r}<$T$1,"Вторая четверть","Больше половины"))'

    # U (col 21): Отбор для поиска (внутри группы)
    ws.cell(row=r, column=21).value = (
        f'=IF(OR(M{r}<SUMIFS(K:K,Q:Q,1,E:E,E{r})/SUMIFS(J:J,Q:Q,1,E:E,E{r})*0.5,'
        f'N{r}<SUMIFS(L:L,Q:Q,1,E:E,E{r})/SUMIFS(K:K,Q:Q,1,E:E,E{r})*0.5),'
        f'"код для проверки","нормальный код")'
    )

    # V (col 22): Отбор по CTR (внутри группы)
    ws.cell(row=r, column=22).value = (
        f'=IF(M{r}<SUMIFS(K:K,Q:Q,1,E:E,E{r})/SUMIFS(J:J,Q:Q,1,E:E,E{r})*0.5,'
        f'"Низшая четверть",IF(M{r}<SUMIFS(K:K,Q:Q,1,E:E,E{r})/SUMIFS(J:J,Q:Q,1,E:E,E{r}),'
        f'"Вторая четверть","Больше половины"))'
    )

    # W (col 23): Отбор по CR (внутри группы)
    ws.cell(row=r, column=23).value = (
        f'=IF(N{r}<SUMIFS(L:L,Q:Q,1,E:E,E{r})/SUMIFS(K:K,Q:Q,1,E:E,E{r})*0.5,'
        f'"Низшая четверть",IF(N{r}<SUMIFS(L:L,Q:Q,1,E:E,E{r})/SUMIFS(K:K,Q:Q,1,E:E,E{r}),'
        f'"Вторая четверть","Больше половины"))'
    )

    # X (col 24): Количество фото (-2 от скрипта)
    ws.cell(row=r, column=24).value = int(row_data['Количество фото (-2 от скрипта)'])

    # === Применяем стили из шаблонной строки 3 ===
    for col in range(1, 25):
        cell = ws.cell(row=r, column=col)
        style = row3_styles.get(col)
        if style:
            cell.font         = copy(style['font'])
            cell.fill         = copy(style['fill'])
            cell.border       = copy(style['border'])
            cell.alignment    = copy(style['alignment'])
            cell.number_format = style['number_format']

# === ОБНОВЛЯЕМ ФОРМУЛЫ В СТРОКЕ 1 (диапазон под новое кол-во строк) ===
ws['J1'] = f'=SUBTOTAL(9,J3:J{last_data_row})'
ws['K1'] = f'=SUBTOTAL(9,K3:K{last_data_row})'
ws['L1'] = f'=SUBTOTAL(9,L3:L{last_data_row})'
ws['M1'] = '=K1/J1'
ws['N1'] = '=L1/K1'
ws['S1'] = '=SUMIF(Q:Q,1,K:K)/SUMIF(Q:Q,1,J:J)'
ws['T1'] = '=SUMIF(Q:Q,1,L:L)/SUMIF(Q:Q,1,K:K)'

# === ОБНОВЛЯЕМ ЗАГОЛОВОК столбца Остаток ===
ws.cell(row=2, column=15).value = f'Остаток на {output_date}'

# === СОХРАНЯЕМ КАК НОВЫЙ ФАЙЛ ===
wb.save(output_path)
print(f'Готово! Файл сохранён: {output_path}')
print(f'Строк данных: {len(df_base)}')
print(f'Диапазон: строки 3–{last_data_row}')

# Проверка
from collections import Counter
техничка = []
for r in range(3, last_data_row + 1):
    техничка.append(ws.cell(row=r, column=17).value)
counts = Counter(техничка)
print(f'Техничка: 1={counts.get(1, 0)}, 0={counts.get(0, 0)}')

Готово! Файл сохранён: Отбор 12.04.xlsx
Строк данных: 1323
Диапазон: строки 3–1325
Техничка: 1=912, 0=411
